In [ ]:
# 02 - Data Profiling & Quality Assessment

## Purpose
This notebook performs systematic profiling of all raw HDX datasets to understand data quality, structure, and potential issues before transformation.

## Why Data Profiling Matters
For humanitarian data work, understanding data quality is critical because:
- **Missing data** can affect population estimates and needs assessments
- **Inconsistent formats** break joins between tables (e.g., admin codes)
- **Outliers** may indicate data entry errors or real humanitarian hotspots

## What This Notebook Does
1. Connects to PostgreSQL database
2. Profiles each table for:
   - Data types and structure
   - Missing values (counts and percentages)
   - Unique values and distinct counts
   - Duplicate records
   - Summary statistics (numeric columns)
3. Documents data quality issues
4. Provides recommendations for cleaning in dbt

## Tables Profiled
- `raw_data.education_facilities` (573 rows)
- `raw_data.health_facilities` (1,988 rows)
- `raw_data.health_facility_type` (1,513 rows)
- `raw_data.jiaf_south_sudan_2026` (239 rows)
- `raw_data.population_estimates_2024` (80 rows)

## Output
A data quality matrix with prioritized issues for the SQL cleaning phase.

In [1]:
import os
import pandas as pd
import numpy as np

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# For better display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Imports loaded successfully")

Imports loaded successfully


In [2]:
# Load environment variables
load_dotenv()

# Database credentials
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

# Create connection string and engine
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), version();"))
    db_name, version = result.fetchone()
    print(f"Connected to: {db_name}")
    print(f"PostgreSQL version: {version.split(',')[0]}")

# List of tables to profile
TABLES = [
    'raw_data.education_facilities',
    'raw_data.health_facilities',
    'raw_data.health_facility_type',
    'raw_data.jiaf_south_sudan_2026',
    'raw_data.population_estimates_2024'
]

print(f"\n {len(TABLES)} tables ready for profiling")

Connected to: humanitarian_db
PostgreSQL version: PostgreSQL 17.10 on x86_64-windows

 5 tables ready for profiling


In [3]:
def get_table_info(table_name):
    """Get basic information about a table."""
    query = f"""
    SELECT 
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'raw_data' 
        AND table_name = '{table_name.split('.')[-1]}'
    ORDER BY ordinal_position;
    """
    with engine.connect() as conn:
        return pd.read_sql(query, conn)

def profile_table(table_name, df):
    """Generate comprehensive profile for a single table."""
    
    # Basic info
    profile = {
        'table': table_name,
        'total_rows': len(df),
        'total_columns': len(df.columns),
        'duplicate_rows': df.duplicated().sum(),
        'missing_cells': df.isnull().sum().sum(),
        'missing_percent': round((df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100, 2)
    }
    
    # Column-level missing data
    missing_df = pd.DataFrame({
        'column': df.columns,
        'data_type': df.dtypes.astype(str).values,
        'missing_count': df.isnull().sum().values,
        'missing_percent': round((df.isnull().sum() / len(df)) * 100, 2).values,
        'unique_values': df.nunique().values
    })
    
    return profile, missing_df

def preview_table(table_name, n=3):
    """Load a preview of the table."""
    query = f"SELECT * FROM {table_name} LIMIT {n};"
    with engine.connect() as conn:
        return pd.read_sql(query, conn)

print(" Profiling functions defined")

 Profiling functions defined


In [5]:
# Dictionary to store all profiling results
profiles = {}
missing_summaries = {}

# Profile each table
for table in TABLES:
    print(f"\n{'='*60}")
    print(f"Profiling: {table}")
    print('='*60)
    
    # Load full table
    query = f"SELECT * FROM {table};"
    df = pd.read_sql(query, engine)
    
    # Get profile
    profile, missing_df = profile_table(table, df)
    profiles[table] = profile
    missing_summaries[table] = missing_df
    
    # Display basic info
    print(f"Rows: {profile['total_rows']:,}")
    print(f"Columns: {profile['total_columns']}")
    print(f"Duplicate rows: {profile['duplicate_rows']:,}")
    print(f"Missing cells: {profile['missing_cells']:,} ({profile['missing_percent']}%)")
    
    # Display missing data summary (top columns with missing values)
    missing_cols = missing_df[missing_df['missing_count'] > 0].sort_values('missing_count', ascending=False)
    
    if len(missing_cols) > 0:
        print(f"\nColumns with missing values ({len(missing_cols)} columns):")
        display(missing_cols[['column', 'data_type', 'missing_count', 'missing_percent']].head(10))
    else:
        print("\nNo missing values found!")
    
    # Display column info
    print(f"\nColumn data types:")
    print(df.dtypes.value_counts().to_string())
    
    # Preview first 2 rows
    print(f"\nPreview (first 2 rows):")
    display(df.head(2))

print(f"\n{'='*60}")
print("All tables profiled successfully!")


Profiling: raw_data.education_facilities
Rows: 573
Columns: 22
Duplicate rows: 0
Missing cells: 5,217 (41.39%)

Columns with missing values (12 columns):


,column,data_type,missing_count,missing_percent
9,source,object,573,100.00
18,adm4_pcode,object,573,100.00
7,addr_full,object,573,100.00
6,capacity_persons,object,573,100.00
19,adm4_name,object,573,100.00
2,name_en,object,568,99.13
5,operator_type,object,491,85.69
8,addr_city,object,472,82.37
4,building,object,462,80.63
1,name,object,135,23.56



Column data types:
object    22

Preview (first 2 rows):


,id,name,name_en,amenity,building,operator_type,capacity_persons,addr_full,addr_city,source,adm0_pcode,adm0_name,adm1_pcode,adm1_name,adm2_pcode,adm2_name,adm3_pcode,adm3_name,adm4_pcode,adm4_name,name_latin,geometry
0,way/974164633,Africano Mande Academy,None,school,None,None,None,None,Maridi,None,SSD,South Sudan,SS10,Western Equatoria,SS1003,Maridi,SS100304,Maridi,None,None,Africano Mande Academy,0103000020E61000000100000011000000A62089A8D373...
1,way/974164631,Maridi Teacher Training Institute,None,school,None,None,None,None,Maridi,None,SSD,South Sudan,SS10,Western Equatoria,SS1003,Maridi,SS100304,Maridi,None,None,Maridi Teacher Training Institute,0103000020E610000001000000070000000A7EC0A84973...



Profiling: raw_data.health_facilities
Rows: 1,988
Columns: 10
Duplicate rows: 0
Missing cells: 917 (4.61%)

Columns with missing values (5 columns):


...


Profiling: raw_data.population_estimates_2024
Rows: 80
Columns: 22
Duplicate rows: 0
Missing cells: 79 (4.49%)

Columns with missing values (5 columns):


,column,data_type,missing_count,missing_percent
3,admin2_alternate_name,object,75,93.75
0,admin1,object,1,1.25
1,admin1_pcode,object,1,1.25
2,admin2,object,1,1.25
4,admin2_pcode,object,1,1.25



Column data types:
float64    17
object      5

Preview (first 2 rows):


,admin1,admin1_pcode,admin2,admin2_alternate_name,admin2_pcode,population_-_2025,%_male_children\n_under_5,no._of_male\nchildren_under_5,%_female\nchildren_under_5,no._of_female\nchildren_under_5,%_male_children_\naged_5_-_17_years,no._of_male_children_\naged_5_-_17_years,%_female_children_\naged_5_-_17_years,no._of_female_\nchildren_aged_5_-_17_years,%_male_adults_\naged_18_-_60,no._of__male_\nadults_aged_18_-_60,%_female_adults_\naged_18_-_60,no._of_female_\nadults_aged_18_-_60,%_male_adults_\naged_over_60,no._of_male_adults_\naged_over_60,%_female_adults_\naged_over_60,no._female_adults_\naged_over_60
0,Abyei Administrative Area,SS00,Abyei Administrative Area,Abyei Administrative Area,SS0001,145358.00,0.10,13954.37,0.09,12936.86,0.18,25728.37,0.17,25146.93,0.19,27618.02,0.20,29362.32,0.03,4942.17,0.04,5668.96
1,Central Equatoria,SS01,Juba,None,SS0101,570834.14,0.07,42241.73,0.10,55599.24,0.15,87908.46,0.16,90762.63,0.23,132433.52,0.24,137571.03,0.02,13414.60,0.02,10902.93



All tables profiled successfully!


In [6]:
# Create a summary dataframe of all profiles
summary_data = []
for table, profile in profiles.items():
    summary_data.append({
        'Table': table.split('.')[-1],
        'Rows': profile['total_rows'],
        'Columns': profile['total_columns'],
        'Duplicates': profile['duplicate_rows'],
        'Missing Cells': profile['missing_cells'],
        'Missing %': profile['missing_percent']
    })

summary_df = pd.DataFrame(summary_data)
print("Data Quality Summary")
print('='*60)
display(summary_df)

# Identify tables with quality issues
print("\nData Quality Flags:")
print('='*60)

flags = []

for table, missing_df in missing_summaries.items():
    table_name = table.split('.')[-1]
    
    # Flag: High missing percentage (>20%)
    high_missing = missing_df[missing_df['missing_percent'] > 20]
    if len(high_missing) > 0:
        for _, row in high_missing.iterrows():
            flags.append({
                'Table': table_name,
                'Issue': f"High missing data: {row['column']} ({row['missing_percent']:.1f}% missing)",
                'Priority': 'High'
            })
    
    # Flag: Duplicate rows
    if profiles[table]['duplicate_rows'] > 0:
        flags.append({
            'Table': table_name,
            'Issue': f"Contains {profiles[table]['duplicate_rows']:,} duplicate rows",
            'Priority': 'Medium'
        })
    
    # Flag: Tables with missing data overall
    if profiles[table]['missing_percent'] > 0:
        flags.append({
            'Table': table_name,
            'Issue': f"Overall {profiles[table]['missing_percent']:.1f}% missing data",
            'Priority': 'Low'
        })

# Display flags as dataframe
flags_df = pd.DataFrame(flags)
if len(flags_df) > 0:
    display(flags_df.sort_values('Priority', ascending=False))
else:
    print("No quality issues detected!")

# Table-specific observations
print("\nTable-Specific Observations:")
print('='*60)

for table in TABLES:
    table_name = table.split('.')[-1]
    profile = profiles[table]
    missing_df = missing_summaries[table]
    
    print(f"\n {table_name}:")
    
    # Find columns with most missing data
    missing_cols = missing_df[missing_df['missing_count'] > 0].sort_values('missing_percent', ascending=False)
    if len(missing_cols) > 0:
        print(f"   - Top missing columns: {', '.join(missing_cols['column'].head(3).tolist())}")
    
    # Data type issues
    dtypes = missing_df['data_type'].value_counts()
    print(f"   - Column types: {dict(dtypes)}")
    
    # Check for key columns
    if 'geometry' in missing_df['column'].values:
        print(f"   - Contains spatial data (geometry column)")
    if 'latitude' in missing_df['column'].values or 'longitude' in missing_df['column'].values:
        print(f"   - Contains geographic coordinates")
    if 'admin1_pcode' in missing_df['column'].values or 'admin2_pcode' in missing_df['column'].values:
        print(f"   - Contains admin hierarchy codes")
    
    # Table-specific notes
    if table_name == 'jiaf_south_sudan_2026':
        print(f"   - Column names need cleaning ('unnamed:_1', etc.)")
    if table_name == 'population_estimates_2024':
        print(f"   - Column names contain newline characters (\\n)")

print("\n" + "="*60)
print(" Data profiling complete!")

Data Quality Summary


,Table,Rows,Columns,Duplicates,Missing Cells,Missing %
0,education_facilities,573,22,0,5217,41.39
1,health_facilities,1988,10,0,917,4.61
2,health_facility_type,1513,11,0,450,2.70
3,jiaf_south_sudan_2026,239,21,0,1040,20.72
4,population_estimates_2024,80,22,0,79,4.49



Data Quality Flags:


,Table,Issue,Priority
11,education_facilities,Overall 41.4% missing data,Low
13,health_facilities,Overall 4.6% missing data,Low
20,jiaf_south_sudan_2026,Overall 20.7% missing data,Low
14,health_facility_type,Overall 2.7% missing data,Low
22,population_estimates_2024,Overall 4.5% missing data,Low
...,...,...,...
10,education_facilities,High missing data: name_latin (23.6% missing),High
9,education_facilities,High missing data: adm4_name (100.0% missing),High
8,education_facilities,High missing data: adm4_pcode (100.0% missing),High
7,education_facilities,High missing data: source (100.0% missing),High



Table-Specific Observations:

 education_facilities:
   - Top missing columns: source, adm4_pcode, addr_full
   - Column types: {'object': np.int64(22)}
   - Contains spatial data (geometry column)

 health_facilities:
   - Top missing columns: site_dhis2_name, longitude, latitude
   - Column types: {'object': np.int64(8), 'float64': np.int64(2)}
   - Contains geographic coordinates

 health_facility_type:
   - Top missing columns: Latitude, Longitude
   - Column types: {'object': np.int64(8), 'float64': np.int64(2), 'int64': np.int64(1)}

 jiaf_south_sudan_2026:
   - Top missing columns: unnamed:_4, unnamed:_5, unnamed:_20
   - Column types: {'object': np.int64(21)}
   - Column names need cleaning ('unnamed:_1', etc.)

 population_estimates_2024:
   - Top missing columns: admin2_alternate_name, admin1, admin1_pcode
   - Column types: {'float64': np.int64(17), 'object': np.int64(5)}
   - Contains admin hierarchy codes
   - Column names contain newline characters (\n)

 Data profiling 

In [7]:
# Create a summary dataframe of all profiles
summary_data = []
for table, profile in profiles.items():
    summary_data.append({
        'Table': table.split('.')[-1],
        'Rows': profile['total_rows'],
        'Columns': profile['total_columns'],
        'Duplicates': profile['duplicate_rows'],
        'Missing Cells': profile['missing_cells'],
        'Missing %': profile['missing_percent']
    })

summary_df = pd.DataFrame(summary_data)
print("Data Quality Summary")
print('='*60)
display(summary_df)

# Identify tables with quality issues
print("\nData Quality Flags (Prioritized):")
print('='*60)

flags = []

for table, missing_df in missing_summaries.items():
    table_name = table.split('.')[-1]
    
    # CRITICAL: 100% missing columns (entire columns empty)
    critical_missing = missing_df[missing_df['missing_percent'] == 100]
    if len(critical_missing) > 0:
        for _, row in critical_missing.iterrows():
            flags.append({
                'Table': table_name,
                'Issue': f"EMPTY COLUMN: {row['column']} (100% missing)",
                'Priority': 'CRITICAL'
            })
    
    # HIGH: Columns with >20% missing
    high_missing = missing_df[(missing_df['missing_percent'] > 20) & (missing_df['missing_percent'] < 100)]
    if len(high_missing) > 0:
        for _, row in high_missing.iterrows():
            flags.append({
                'Table': table_name,
                'Issue': f"High missing: {row['column']} ({row['missing_percent']:.1f}% missing)",
                'Priority': 'HIGH'
            })
    
    # MEDIUM: Duplicate rows
    if profiles[table]['duplicate_rows'] > 0:
        flags.append({
            'Table': table_name,
            'Issue': f"Contains {profiles[table]['duplicate_rows']:,} duplicate rows",
            'Priority': 'MEDIUM'
        })
    
    # MEDIUM: Column name issues
    if table_name == 'jiaf_south_sudan_2026':
        flags.append({
            'Table': table_name,
            'Issue': "Column names need cleaning ('unnamed:_1', etc.)",
            'Priority': 'HIGH'
        })
    
    if table_name == 'population_estimates_2024':
        flags.append({
            'Table': table_name,
            'Issue': "Column names contain newline characters (\\n) - needs cleaning",
            'Priority': 'MEDIUM'
        })

# Display flags sorted by priority
flags_df = pd.DataFrame(flags)
priority_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3}
if len(flags_df) > 0:
    flags_df['sort_order'] = flags_df['Priority'].map(priority_order)
    display(flags_df.sort_values('sort_order')[['Table', 'Issue', 'Priority']])
else:
    print("No quality issues detected!")

# Table-specific observations
print("\n📝 Table-Specific Observations:")
print('='*60)

for table in TABLES:
    table_name = table.split('.')[-1]
    profile = profiles[table]
    missing_df = missing_summaries[table]
    
    print(f"\n{table_name}:")
    
    # Find columns with most missing data (excluding 100% empty columns)
    missing_cols = missing_df[(missing_df['missing_count'] > 0) & (missing_df['missing_percent'] < 100)]
    missing_cols = missing_cols.sort_values('missing_percent', ascending=False)
    
    if len(missing_cols) > 0:
        print(f"   - Top missing columns: {', '.join(missing_cols['column'].head(3).tolist())}")
    
    # Empty columns
    empty_cols = missing_df[missing_df['missing_percent'] == 100]['column'].tolist()
    if empty_cols:
        print(f"   -  EMPTY COLUMNS: {', '.join(empty_cols)}")
    
    # Data type summary
    dtypes = missing_df['data_type'].value_counts()
    print(f"   - Column types: {dict(dtypes)}")
    
    # Key columns present
    if 'geometry' in missing_df['column'].values:
        print(f"   - Contains spatial data (geometry column)")
    if 'latitude' in missing_df['column'].values or 'longitude' in missing_df['column'].values:
        print(f"   - Contains geographic coordinates")
    if 'admin1_pcode' in missing_df['column'].values or 'admin2_pcode' in missing_df['column'].values:
        print(f"   - Contains admin hierarchy codes")
    
    # Table-specific notes
    if table_name == 'jiaf_south_sudan_2026':
        print(f"   -   Column names need cleaning (contains 'unnamed:_1', etc.)")
    if table_name == 'population_estimates_2024':
        print(f"   -   Column names contain newline characters (\\n)")

print("\n" + "="*60)
print(" Data profiling complete!")

# Summary of critical issues
print("\n CRITICAL ISSUES IDENTIFIED:")
critical_issues = flags_df[flags_df['Priority'] == 'CRITICAL']
if len(critical_issues) > 0:
    for _, row in critical_issues.iterrows():
        print(f"   - {row['Table']}: {row['Issue']}")
else:
    print("    No critical issues found")


Data Quality Summary


,Table,Rows,Columns,Duplicates,Missing Cells,Missing %
0,education_facilities,573,22,0,5217,41.39
1,health_facilities,1988,10,0,917,4.61
2,health_facility_type,1513,11,0,450,2.70
3,jiaf_south_sudan_2026,239,21,0,1040,20.72
4,population_estimates_2024,80,22,0,79,4.49



Data Quality Flags (Prioritized):


,Table,Issue,Priority
0,education_facilities,EMPTY COLUMN: capacity_persons (100% missing),CRITICAL
1,education_facilities,EMPTY COLUMN: addr_full (100% missing),CRITICAL
2,education_facilities,EMPTY COLUMN: source (100% missing),CRITICAL
3,education_facilities,EMPTY COLUMN: adm4_pcode (100% missing),CRITICAL
4,education_facilities,EMPTY COLUMN: adm4_name (100% missing),CRITICAL
5,education_facilities,High missing: name (23.6% missing),HIGH
6,education_facilities,High missing: name_en (99.1% missing),HIGH
7,education_facilities,High missing: building (80.6% missing),HIGH
8,education_facilities,High missing: operator_type (85.7% missing),HIGH
9,education_facilities,High missing: addr_city (82.4% missing),HIGH



📝 Table-Specific Observations:

education_facilities:
   - Top missing columns: name_en, operator_type, addr_city
   -  EMPTY COLUMNS: capacity_persons, addr_full, source, adm4_pcode, adm4_name
   - Column types: {'object': np.int64(22)}
   - Contains spatial data (geometry column)

health_facilities:
   - Top missing columns: site_dhis2_name, longitude, latitude
   - Column types: {'object': np.int64(8), 'float64': np.int64(2)}
   - Contains geographic coordinates

health_facility_type:
   - Top missing columns: Latitude, Longitude
   - Column types: {'object': np.int64(8), 'float64': np.int64(2), 'int64': np.int64(1)}

jiaf_south_sudan_2026:
   - Top missing columns: unnamed:_4, unnamed:_5, unnamed:_20
   - Column types: {'object': np.int64(21)}
   -   Column names need cleaning (contains 'unnamed:_1', etc.)

population_estimates_2024:
   - Top missing columns: admin2_alternate_name, admin1, admin1_pcode
   - Column types: {'float64': np.int64(17), 'object': np.int64(5)}
   - Contai